## Geoginah Ramatshe_26303686

# 3. Model Training

This notebook trains a Logistic Regression baseline and, when the `xgboost` package is installed, tunes three modest XGBoost configurations using the validation subset. It saves fitted model pipelines and training/validation probabilities for the validation and visualisation notebooks.

## Hyperparameter choices and rationale

**Logistic Regression baseline**
- `C=1.0`: standard moderate regularisation; smaller values apply stronger regularisation.
- `penalty="l2"`: discourages excessively large coefficients.
- `solver="liblinear"`: supports binary classification with L2 regularisation.
- `class_weight="balanced"`: increases the weight of the minority class based on training labels.
- `max_iter=2000`: allows more iterations for convergence after categorical encoding.
- `random_state=42`: supports reproducibility.

**XGBoost candidates (optional)**
- `n_estimators` of 200–300 and `learning_rate` of 0.04–0.05 provide gradual boosting across a moderate number of trees.
- `max_depth` of 2–4 limits tree complexity; shallow trees help control overfitting.
- `min_child_weight`, row/column subsampling and L1/L2 regularisation help constrain complex splits.
- `scale_pos_weight` is calculated from the training class ratio to account for class imbalance.
- The best candidate is selected by validation PR-AUC, which is informative when the dormant class is less common.

All imputation, scaling and one-hot encoding are inside the pipeline and are fitted only on training data. The test subset is not used to select hyperparameters.

In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Raw CSV files are expected to be in the same folder as these notebooks.
BASE_DIR = Path.cwd()
RAW_FILES = { "profile": "table1_customer_profile.csv",
             "funding": "table2_account_funding.csv",
             "transactions": "table3_transaction_trading.csv",
             "engagement": "table4_platform_engagement.csv",}

def safe_date(series):
    return pd.to_datetime(series, errors="coerce")

missing_files = [name for name in RAW_FILES.values() if not (BASE_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Could not find these CSV files in the notebook's current folder: "
        + ", ".join(missing_files)
        + ". Place the CSV files beside the notebooks or update BASE_DIR.")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,roc_auc_score, average_precision_score, confusion_matrix,classification_report)
import joblib

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "processed"
ARTIFACT_DIR = BASE_DIR / "model_artifacts"
RESULTS_DIR = BASE_DIR / "results"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required = [
    PROCESSED_DIR / "train_data.csv",
    PROCESSED_DIR / "validation_data.csv",
    PROCESSED_DIR / "test_data.csv",]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Prepared data not found. Run 02_dataset_preprocessing.ipynb first. Missing: "
        + ", ".join(missing))

train_data = pd.read_csv(PROCESSED_DIR / "train_data.csv", parse_dates=["snapshot_date"])
validation_data = pd.read_csv(PROCESSED_DIR / "validation_data.csv", parse_dates=["snapshot_date"])
test_data = pd.read_csv(PROCESSED_DIR / "test_data.csv", parse_dates=["snapshot_date"])

target_column = "dormancy_target"
excluded_columns = {
    "customer_id", "snapshot_date", target_column,
    "active_investor", "dormant_investor", "days_since_last_activity",
    "registration_date", "registration_dt", "first_deposit_date", "first_deposit_dt",
    "first_deposit_amount", "last_deposit_date", "total_deposit_amount",
    "number_of_deposits", "funding_status", "kyc_completed",}

feature_columns = [c for c in train_data.columns if c not in excluded_columns]

X_train = train_data[feature_columns].copy()
y_train = train_data[target_column].astype(int).copy()
X_val = validation_data[feature_columns].copy()
y_val = validation_data[target_column].astype(int).copy()
X_test = test_data[feature_columns].copy()
y_test = test_data[target_column].astype(int).copy()

# Standardise missing values in text columns for scikit-learn imputers.
for frame in (X_train, X_val, X_test):
    for col in frame.select_dtypes(include=["object", "string", "category"]).columns:
        frame[col] = frame[col].astype(object).where(frame[col].notna(), np.nan)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()

assert "dormant_investor" not in feature_columns
assert "active_investor" not in feature_columns
assert "days_since_last_activity" not in feature_columns

def make_preprocessor():
    numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                                       ("scaler", StandardScaler()),])
    
    categorical_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
                                           ("onehot", OneHotEncoder(handle_unknown="ignore")),])
    
    return ColumnTransformer(transformers=[("numeric", numeric_pipeline, numeric_features),
                                           ("categorical", categorical_pipeline, categorical_features),], remainder="drop")

def compute_metrics(y_true, y_probability, threshold=0.50):
    y_probability = np.asarray(y_probability)
    y_prediction = (y_probability >= threshold).astype(int)
    return {
        "instances": int(len(y_true)),
        "dormant_proportion": float(np.mean(y_true)),
        "accuracy": float(accuracy_score(y_true, y_prediction)),
        "precision": float(precision_score(y_true, y_prediction, zero_division=0)),
        "recall": float(recall_score(y_true, y_prediction, zero_division=0)),
        "f1": float(f1_score(y_true, y_prediction, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_probability)) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": float(average_precision_score(y_true, y_probability)),}

## 1. Train Logistic Regression

In [3]:
logistic_parameters = {"C": 1.0, "penalty": "l2", "solver": "liblinear","class_weight": "balanced", "max_iter": 2000, "random_state": 42,}

logistic_model = Pipeline(steps=[ ("preprocessing", make_preprocessor()), ("classifier", LogisticRegression(**logistic_parameters)),])

logistic_model.fit(X_train, y_train)
logistic_train_probability = logistic_model.predict_proba(X_train)[:, 1]
logistic_validation_probability = logistic_model.predict_proba(X_val)[:, 1]

print("Logistic Regression trained.")
print("Training PR-AUC:", round(compute_metrics(y_train, logistic_train_probability)["pr_auc"], 4))
print("Validation PR-AUC:", round(compute_metrics(y_val, logistic_validation_probability)["pr_auc"], 4))

joblib.dump(logistic_model, ARTIFACT_DIR / "logistic_model.joblib")

Logistic Regression trained.
Training PR-AUC: 0.2791
Validation PR-AUC: 0.4665


['C:\\Users\\HP\\Documents\\4.CAPSTONE Project\\SS3 final\\retail_investor_dormancy_notebooks\\model_artifacts\\logistic_model.joblib']

## 2. Train and select XGBoost


In [4]:
xgb_available = True
try:
    from xgboost import XGBClassifier
except ImportError:
    xgb_available = False
    print("XGBoost is not installed. Skipping XGBoost; Logistic Regression is available.")
    print("To enable XGBoost, run: %pip install xgboost")

xgb_tuning_rows = []
xgb_validation_probability = None
xgb_train_probability = None
best_configuration = None

if xgb_available:
    xgb_candidates = [
        {"configuration": "Conservative shallow trees", "n_estimators": 200, "learning_rate": 0.05, "max_depth": 2,
        "min_child_weight": 5, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.50, "reg_lambda": 2.00,},
        
        {"configuration": "Moderate trees","n_estimators": 250, "learning_rate": 0.04, "max_depth": 3,
        "min_child_weight": 3, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.20, "reg_lambda": 2.00,},
        
        {"configuration": "Deeper trees", "n_estimators": 300, "learning_rate": 0.05, "max_depth": 4,
        "min_child_weight": 1, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.10, "reg_lambda": 1.00,},
    ]
    
    positive = max(int(y_train.sum()), 1)
    negative = int((y_train == 0).sum())
    scale_pos_weight = negative / positive
    xgb_validation_models = {}

    for candidate in xgb_candidates:
        params = {k: v for k, v in candidate.items() if k != "configuration"}
        params.update({"objective": "binary:logistic", "eval_metric": "logloss", "scale_pos_weight": scale_pos_weight, "random_state": 42, "n_jobs": 4,})
        
        candidate_model = Pipeline(steps=[("preprocessing", make_preprocessor()), ("classifier", XGBClassifier(**params)),])
        
        candidate_model.fit(X_train, y_train)
        val_prob = candidate_model.predict_proba(X_val)[:, 1]
        metrics = compute_metrics(y_val, val_prob)
        
        xgb_tuning_rows.append({ **candidate, "validation_pr_auc": metrics["pr_auc"],
            "validation_roc_auc": metrics["roc_auc"], "validation_f1": metrics["f1"],})
        
        xgb_validation_models[candidate["configuration"]] = {"model": candidate_model, "val_probability": val_prob, "params": params,}

    xgb_tuning_results = pd.DataFrame(xgb_tuning_rows).sort_values("validation_pr_auc", ascending=False).reset_index(drop=True)
    display(xgb_tuning_results)
    best_configuration = xgb_tuning_results.iloc[0]["configuration"]
    best_xgb_validation = xgb_validation_models[best_configuration]["model"]
    xgb_validation_probability = xgb_validation_models[best_configuration]["val_probability"]
    xgb_train_probability = best_xgb_validation.predict_proba(X_train)[:, 1]
    print("Selected XGBoost configuration:", best_configuration)
    print("Validation PR-AUC:", round(compute_metrics(y_val, xgb_validation_probability)["pr_auc"], 4))
    joblib.dump(best_xgb_validation, ARTIFACT_DIR / "xgboost_validation_model.joblib")
    xgb_tuning_results.to_csv(RESULTS_DIR / "xgboost_tuning_results.csv", index=False)

,configuration,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,validation_pr_auc,validation_roc_auc,validation_f1
0,Conservative shallow trees,200,0.05,2,5,0.8,0.8,0.5,2.0,0.401577,0.759599,0.447384
1,Moderate trees,250,0.04,3,3,0.8,0.8,0.2,2.0,0.382402,0.744467,0.459016
2,Deeper trees,300,0.05,4,1,0.8,0.8,0.1,1.0,0.344656,0.710787,0.342321


Selected XGBoost configuration: Conservative shallow trees
Validation PR-AUC: 0.4016


## 3. Save training-stage metrics and probabilities

In [5]:
def metrics_row(model_name, split_name, y_true, probability):
    return {"model": model_name, "split": split_name, **compute_metrics(y_true, probability)}

training_rows = [metrics_row("Logistic Regression", "Training", y_train, logistic_train_probability),
                 metrics_row("Logistic Regression", "Validation", y_val, logistic_validation_probability),]

probability_data = pd.DataFrame({
    "split": ["Training"] * len(y_train) + ["Validation"] * len(y_val),
    "actual": np.concatenate([y_train.to_numpy(), y_val.to_numpy()]),
    "logistic_probability": np.concatenate([logistic_train_probability, logistic_validation_probability]),
})

if xgb_available:
    training_rows.extend([metrics_row("XGBoost", "Training", y_train, xgb_train_probability),
                          metrics_row("XGBoost", "Validation", y_val, xgb_validation_probability),])
    
    probability_data["xgboost_probability"] = np.concatenate([xgb_train_probability, xgb_validation_probability])

training_metrics = pd.DataFrame(training_rows)
display(training_metrics.round(4))
training_metrics.to_csv(RESULTS_DIR / "training_validation_metrics.csv", index=False)
probability_data.to_csv(RESULTS_DIR / "training_validation_probabilities.csv", index=False)
print("Saved models to:", ARTIFACT_DIR.resolve())
print("Saved training/validation outputs to:", RESULTS_DIR.resolve())

,model,split,instances,dormant_proportion,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression,Training,2138,0.0800,0.7306,0.1918,0.7368,0.3043,0.8076,0.2791
1,Logistic Regression,Validation,2553,0.1853,0.5778,0.2810,0.8203,0.4186,0.7607,0.4665
2,XGBoost,Training,2138,0.0800,0.8251,0.2916,0.8304,0.4316,0.9144,0.5270
3,XGBoost,Validation,2553,0.1853,0.6400,0.3126,0.7865,0.4474,0.7596,0.4016


Saved models to: C:\Users\HP\Documents\4.CAPSTONE Project\SS3 final\retail_investor_dormancy_notebooks\model_artifacts
Saved training/validation outputs to: C:\Users\HP\Documents\4.CAPSTONE Project\SS3 final\retail_investor_dormancy_notebooks\results
